In [ ]:
import os
import math
import time
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from tokenizers import ByteLevelBPETokenizer
from google.colab import drive

# ----------------------------------------------------
# 1. Validação de Ambiente e Montagem do Drive
# ----------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

assert torch.cuda.is_available(), (
    "GPU não detectada! Altere o ambiente: "
    "Ambiente de execução -> Alterar tipo de ambiente de execução -> T4 GPU."
)
device = 'cuda'
print(f"Ambiente validado: {device} ({torch.cuda.get_device_name(0)})")

# ----------------------------------------------------
# 2. Definição Estrita de Caminhos no Google Drive
# ----------------------------------------------------
pasta_base = '/content/drive/MyDrive/llm_data'
caminho_train = os.path.join(pasta_base, 'train.bin')
caminho_val = os.path.join(pasta_base, 'val.bin')
pasta_tokenizer = os.path.join(pasta_base, 'tokenizer')
caminho_checkpoint = os.path.join(pasta_base, 'minha_llm_best.pt')

for p, nome in [(caminho_train, "train.bin"), (caminho_val, "val.bin")]:
    assert os.path.exists(p), f"Arquivo obrigatório não encontrado: {nome} em {p}"

# ----------------------------------------------------
# 3. Hiperparâmetros do Modelo e Treinamento
# ----------------------------------------------------
batch_size = 32            # Sequências paralelas por passo
block_size = 256           # Contexto temporal (tokens)
max_iters = 3500           # 3.500 iterações (~13 minutos na T4)
eval_interval = 250        # Avalia loss de validação a cada 250 passos
eval_iters = 25            # Batches para estimativa de perda
learning_rate = 5e-4       # Taxa máxima
min_lr = 5e-5              # Taxa mínima no fim do cosine decay
warmup_iters = 150         # Passos de aquecimento linear do LR

n_embd = 384               # Dimensão de embedding
n_head = 6                 # Cabeças de atenção (64 dims cada)
n_layer = 6                # Blocos Transformer
dropout = 0.2              # Regularização contra overfitting
vocab_size = 4096          # Vocabulário do tokenizador BPE

# ----------------------------------------------------
# 4. Leitura Otimizada de Dados (np.memmap)
# ----------------------------------------------------
train_data = np.memmap(caminho_train, dtype=np.uint16, mode='r')
val_data = np.memmap(caminho_val, dtype=np.uint16, mode='r')

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)

# ----------------------------------------------------
# 5. Arquitetura Transformer Causal Decoder-Only
# ----------------------------------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=None,
            dropout_p=dropout if self.training else 0.0,
            is_causal=True
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP()

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(vocab_size, n_embd),
            wpe = nn.Embedding(block_size, n_embd),
            drop = nn.Dropout(dropout),
            h = nn.ModuleList([Block() for _ in range(n_layer)]),
            ln_f = nn.LayerNorm(n_embd),
        ))
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            return self.lm_head(x[:, [-1], :]), None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.75, top_k=30):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ----------------------------------------------------
# 6. Instanciação e Otimizador
# ----------------------------------------------------
model = GPT().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Modelo instanciado com {total_params / 1e6:.2f}M parâmetros.")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=1e-1)

def get_lr(it):
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    if it > max_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# ----------------------------------------------------
# 7. Loop de Treinamento Definitivo
# ----------------------------------------------------
print(f"\nIniciando treinamento unificado no '{os.path.basename(caminho_checkpoint)}'...")
best_val_loss = float('inf')
t0 = time.time()

for iter_num in range(max_iters + 1):
    lr = get_lr(iter_num)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    if iter_num % eval_interval == 0:
        losses = estimate_loss()
        tempo = time.time() - t0
        print(f"Passo {iter_num:>4}/{max_iters} | Treino Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f} | LR: {lr:.2e} | Tempo: {tempo:.1f}s")

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save({
                'model_state_dict': model.state_dict(),
                'val_loss': best_val_loss,
                'iter': iter_num,
                'config': {'n_embd': n_embd, 'n_head': n_head, 'n_layer': n_layer, 'block_size': block_size, 'vocab_size': vocab_size}
            }, caminho_checkpoint)

    if iter_num == max_iters:
        break

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

print(f"\nTreino concluído com sucesso!")
print(f"Melhor Loss de Validação: {best_val_loss:.4f}")
print(f"Modelo final salvo em:    {caminho_checkpoint}")

# ----------------------------------------------------
# 8. Teste de Inferência nas 4 Dimensões Analíticas
# ----------------------------------------------------
print("\n" + "=" * 65)
print("TESTE DE GERAÇÃO TEXTUAL (4 DIMENSÕES UNIFICADAS)")
print("=" * 65)

tokenizer = ByteLevelBPETokenizer(
    os.path.join(pasta_tokenizer, "vocab.json"),
    os.path.join(pasta_tokenizer, "merges.txt")
)

prompts_teste = [
    ("Copom / Taxa Selic", "O Copom decidiu elevar a taxa Selic"),
    ("Macro / Inflação", "No cenário de referência para a inflação"),
    ("Estabilidade / Risco Sistêmico", "Em relação ao risco de crédito e à estabilidade do sistema financeiro"),
    ("Atuária / Provisões SUSEP", "As provisões técnicas e os limites de solvência")
]

model.eval()
for categoria, prompt in prompts_teste:
    ids = torch.tensor(tokenizer.encode(prompt).ids, dtype=torch.long, device=device).unsqueeze(0)
    saida = model.generate(ids, max_new_tokens=90, temperature=0.75, top_k=30)[0].tolist()
    print(f"\n[{categoria}]")
    print(f"Prompt: '{prompt}'")
    print(f"Continuação: {tokenizer.decode(saida)}\n" + "-" * 55)

Ambiente validado: cuda (Tesla T4)
Modelo instanciado com 12.30M parâmetros.

Iniciando treinamento unificado no 'minha_llm_best.pt'...
Passo    0/3500 | Treino Loss: 8.3565 | Val Loss: 8.3571 | LR: 0.00e+00 | Tempo: 14.2s
Passo  250/3500 | Treino Loss: 4.5140 | Val Loss: 4.4651 | LR: 4.99e-04 | Tempo: 69.2s
Passo  500/3500 | Treino Loss: 3.7073 | Val Loss: 3.7015 | LR: 4.88e-04 | Tempo: 130.5s
Passo  750/3500 | Treino Loss: 3.2777 | Val Loss: 3.3034 | LR: 4.65e-04 | Tempo: 190.2s
Passo 1000/3500 | Treino Loss: 2.9881 | Val Loss: 3.0202 | LR: 4.32e-04 | Tempo: 250.4s
Passo 1250/3500 | Treino Loss: 2.8705 | Val Loss: 2.8584 | LR: 3.91e-04 | Tempo: 310.4s
Passo 1500/3500 | Treino Loss: 2.6379 | Val Loss: 2.6917 | LR: 3.43e-04 | Tempo: 370.7s
Passo 1750/3500 | Treino Loss: 2.5405 | Val Loss: 2.5919 | LR: 2.91e-04 | Tempo: 431.3s
Passo 2000/3500 | Treino Loss: 2.4688 | Val Loss: 2.4907 | LR: 2.38e-04 | Tempo: 491.6s
Passo 2250/3500 | Treino Loss: 2.3787 | Val Loss: 2.4468 | LR: 1.88e-04 | 

In [9]:
import os
import torch
import torch.nn as nn
from torch.nn import functional as F
from tokenizers import ByteLevelBPETokenizer
from google.colab import drive

# 1. Blindagem Obrigatória de Ambiente e Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

!pip install -q gradio

import gradio as gr

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Ambiente validado: {device} ({torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'})")

pasta_base = '/content/drive/MyDrive/llm_data'
pasta_tokenizer = os.path.join(pasta_base, 'tokenizer')
caminho_base_model = os.path.join(pasta_base, 'minha_llm_best.pt')

vocab_path = os.path.join(pasta_tokenizer, "vocab.json")
merges_path = os.path.join(pasta_tokenizer, "merges.txt")

assert os.path.exists(caminho_base_model), f"Checkpoint base não encontrado: {caminho_base_model}"

# 2. Definição Completa da Arquitetura GPT (Auto-contida)
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
vocab_size = 4096
block_size = 256

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.resid_dropout(self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C)))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP()
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(vocab_size, n_embd),
            wpe = nn.Embedding(block_size, n_embd),
            drop = nn.Dropout(dropout),
            h = nn.ModuleList([Block() for _ in range(n_layer)]),
            ln_f = nn.LayerNorm(n_embd),
        ))
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx):
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        return self.lm_head(x[:, [-1], :])

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.6, top_k=25, repetition_penalty=1.15):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
            logits = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            # Penalidade leve e estável contra repetições
            for token_id in set(idx[0].tolist()):
                if logits[0, token_id] > 0:
                    logits[0, token_id] /= repetition_penalty
                else:
                    logits[0, token_id] *= repetition_penalty

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# 3. Carrega o Tokenizador e o Modelo Pré-Treinado Puro
tokenizer = ByteLevelBPETokenizer(vocab_path, merges_path)

model = GPT().to(device)
checkpoint = torch.load(caminho_base_model, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Modelo pré-treinado puro carregado com sucesso (Val Loss de origem: {checkpoint.get('val_loss', 2.29):.4f})")

# ----------------------------------------------------
# 4. Motor de Prompt Wrapping (Tradutor de Intenção)
# ----------------------------------------------------
def converter_pergunta_para_prefixo(pergunta):
    q = pergunta.strip().lower()

    # 1. Risco de Crédito, Inadimplência e Spread
    if any(k in q for k in ['crédito', 'credito', 'inadimplência', 'inadimplencia', 'spread', 'bancos']):
        return "Em relação às condições de crédito no Sistema Financeiro Nacional, as taxas cobradas e o spread bancário refletem o risco de crédito na medida em que"

    # 2. Copom e Taxa Selic
    elif any(k in q for k in ['selic', 'copom', 'juros', 'política monetária', 'politica monetaria']):
        return "O Copom avalia que as decisões de política monetária sobre a taxa Selic impactam a inflação e a economia porque"

    # 3. Inflação e Preços (IPCA / IGP / Monitorados)
    elif any(k in q for k in ['inflação', 'inflacao', 'preço', 'precos', 'ipca', 'igp']):
        return "No cenário de referência para a inflação, o comportamento dos preços livres e dos preços administrados por contrato indica que"

    # 4. Atuária, Provisões e SUSEP
    elif any(k in q for k in ['provisão', 'provisao', 'provisões', 'provisoes', 'solvência', 'solvencia', 'susep', 'seguro', 'sinistro']):
        return "No âmbito atuarial e regulatório da SUSEP, a constituição de provisões técnicas e os limites de solvência são essenciais porque"

    # 5. Fallback Inteligente (Transforma pergunta genérica em prefixo afirmativo)
    else:
        q_limpa = pergunta.replace('?', '').replace('Como', '').replace('como', '').replace('Qual', '').replace('qual', '').strip()
        return f"No contexto econômico e financeiro brasileiro, {q_limpa} ocorre porque"

# ----------------------------------------------------
# 5. Função de Resposta para a Interface
# ----------------------------------------------------
def responder_usuario(pergunta_usuario, modo_livre, tamanho_resposta, temperatura):
    if not pergunta_usuario.strip():
        return "Por favor, digite uma pergunta ou um texto inicial.", ""

    if modo_livre:
        prefixo_usado = pergunta_usuario
    else:
        prefixo_usado = converter_pergunta_para_prefixo(pergunta_usuario)

    ids = torch.tensor(tokenizer.encode(prefixo_usado).ids, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        saida_ids = model.generate(
            ids,
            max_new_tokens=int(tamanho_resposta),
            temperature=float(temperatura),
            top_k=25,
            repetition_penalty=1.15
        )[0].tolist()

    texto_gerado = tokenizer.decode(saida_ids)
    return texto_gerado, prefixo_usado

# ----------------------------------------------------
# 6. Interface Gráfica Interativa
# ----------------------------------------------------
with gr.Blocks(title="Macro LLM - BCB & Atuária (Ambiente Seguro)") as demo:
    gr.Markdown("# 🏛️ Macro LLM: Sistema Financeiro, Política Monetária e Atuária")
    gr.Markdown(
        "Interface analítica operando sobre o **modelo causal de 12,3M de parâmetros** pré-treinado sobre as Atas do Copom, "
        "Relatórios de Política Monetária (RPM), Relatórios de Estabilidade Financeira (REF) e normas da SUSEP."
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_texto = gr.Textbox(
                label="Digite sua pergunta ou tema:",
                placeholder="Ex: Como o risco de crédito afeta a taxa de juros?",
                lines=4
            )
            check_modo_livre = gr.Checkbox(
                label="Modo Livre (Desativa o Prompt Wrapper e gera continuação exata do seu texto)",
                value=False
            )
            with gr.Accordion("Parâmetros de Amostragem", open=False):
                slider_tokens = gr.Slider(minimum=40, maximum=200, value=90, step=10, label="Tamanho da Resposta (tokens)")
                slider_temp = gr.Slider(minimum=0.2, maximum=1.0, value=0.6, step=0.05, label="Temperatura (0.6 = Estável)")

            btn_enviar = gr.Button("Gerar Análise", variant="primary")

        with gr.Column(scale=1):
            output_resposta = gr.Textbox(label="Análise Gerada pelo Modelo:", lines=9)
            output_prefixo = gr.Textbox(label="Prefixo Técnico Aplicado (Prompt Wrapper):", lines=2)

    btn_enviar.click(
        fn=responder_usuario,
        inputs=[input_texto, check_modo_livre, slider_tokens, slider_temp],
        outputs=[output_resposta, output_prefixo]
    )

demo.launch(share=True)

Ambiente validado: cuda (Tesla T4)
Modelo pré-treinado puro carregado com sucesso (Val Loss de origem: 2.2903)
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7f8984327e2a341bfd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
